In [1]:
import os
import numpy as np
import pandas as pd

In [2]:

# Path matching your existing experiment setup
FILEPATH = "data/Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv"

assert os.path.exists(FILEPATH), f"File not found at {FILEPATH}. Check your path."

print(f"[*] Ingesting raw CSV file directly from: {FILEPATH}")
# Read without any initial type-casting or conversions
df_raw = pd.read_csv(FILEPATH, encoding="latin-1", low_memory=False)

# Strip whitespace from raw column names immediately to avoid hidden string mismatch bugs
df_raw.columns = df_raw.columns.str.strip()

print(f"[+] Successfully loaded raw file.")
print(f"    -> Raw Rows:    {df_raw.shape[0]:,}")
print(f"    -> Raw Columns: {df_raw.shape[1]:,}")

[*] Ingesting raw CSV file directly from: data/Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
[+] Successfully loaded raw file.
    -> Raw Rows:    458,968
    -> Raw Columns: 85


In [3]:
# 1. Verify target label presence
if "Label" not in df_raw.columns:
    raise ValueError("Critical Error: 'Label' column not detected in raw headers.")

# 2. Check for duplicate header rows accidentally ingested as data rows
# In raw CICIDS files, repeated headers often contain the column title 'Destination Port' or 'Label'
header_leak_mask = (df_raw["Label"].astype(str).str.strip() == "Label") | (
    df_raw.iloc[:, 0].astype(str).str.strip() == df_raw.columns[0]
)
duplicate_header_count = header_leak_mask.sum()

print("==================== STEP 1 SCHEMA AUDIT ====================")
print(f"Total raw observations:           {len(df_raw):,}")
print(f"Total raw features/columns:       {len(df_raw.columns):,}")
print(f"Repeated internal header rows:    {duplicate_header_count:,}")

# 3. Check for entirely empty rows or duplicate physical rows
empty_rows = df_raw.isna().all(axis=1).sum()
duplicate_rows = df_raw.duplicated().sum()
print(f"Completely empty rows:            {empty_rows:,}")
print(f"Exact duplicated rows:            {duplicate_rows:,}")

# 4. Check presence of expected network identifier metadata columns
expected_metadata = [
    "Flow ID",
    "Source IP",
    "Source Port",
    "Destination IP",
    "Destination Port",
    "Timestamp",
]
found_metadata = [col for col in expected_metadata if col in df_raw.columns]
print(f"Detected identifier columns:      {found_metadata}")
print("=============================================================")

==================== STEP 1 SCHEMA AUDIT ====================
Total raw observations:           458,968
Total raw features/columns:       85
Repeated internal header rows:    0
Completely empty rows:            288,602
Exact duplicated rows:            288,602
Detected identifier columns:      ['Flow ID', 'Source IP', 'Source Port', 'Destination IP', 'Destination Port', 'Timestamp']


In [4]:
# Profile raw class breakdown directly before any transformations or conversions
raw_label_counts = df_raw["Label"].value_counts(dropna=False)
raw_label_percent = df_raw["Label"].value_counts(dropna=False, normalize=True) * 100

df_raw_distribution = pd.DataFrame(
    {"Count": raw_label_counts, "Percentage (%)": raw_label_percent.round(4)}
)

print("=================== RAW LABEL DISTRIBUTION ===================")
display(df_raw_distribution)
print("==============================================================")

=================== RAW LABEL DISTRIBUTION ===================


,Count,Percentage (%)
Label,,
NaN,288602,62.8806
BENIGN,168186,36.6444
Web Attack  Brute Force,1507,0.3283
Web Attack  XSS,652,0.1421
Web Attack  Sql Injection,21,0.0046


In [5]:
# Create a working copy before conversion to track changes exactly
df_audit = df_raw.copy()

# Drop the network identifier metadata first, matching your pipeline
metadata_cols = ['Flow ID', 'Source IP', 'Source Port', 'Destination IP', 'Destination Port', 'Timestamp']
cols_dropped = [col for col in metadata_cols if col in df_audit.columns]
df_audit.drop(columns=cols_dropped, inplace=True)

# 1. Count NaNs present in the raw data per column BEFORE any numeric casting
raw_nans_per_col = df_audit.isna().sum()
total_raw_nans = raw_nans_per_col.sum()

# 2. Search for string representations of Infinity across the DataFrame
infinity_strings = ["Infinity", "infinity", "Inf", "-Infinity", "-infinity"]
inf_string_counts = {}
for col in df_audit.columns:
    if col != 'Label':
        # Count occurrences of literal inf strings
        inf_mask = df_audit[col].astype(str).str.strip().isin(infinity_strings)
        count = inf_mask.sum()
        if count > 0:
            inf_string_counts[col] = count

print("==================== STEP 2 MISSINGNESS & INF STRINGS ====================")
print(f"Total NaNs before numeric conversion:            {total_raw_nans:,}")
print(f"Columns containing literal 'Infinity' strings:    {len(inf_string_counts)}")
for col, cnt in inf_string_counts.items():
    print(f"  -> {col}: {cnt:,} occurrences")
print("==========================================================================")

==================== STEP 2 MISSINGNESS & INF STRINGS ====================
Total NaNs before numeric conversion:            22,799,578
Columns containing literal 'Infinity' strings:    0


In [7]:
# Profile what happens when applying pd.to_numeric(..., errors='coerce')
coerced_nan_counts = {}
coerced_sample_values = {}

feature_cols = [col for col in df_audit.columns if col != 'Label']

for col in feature_cols:
    # Identify non-null values prior to coercion
    was_not_null = df_audit[col].notna()
    
    # Coerce to numeric
    converted_series = pd.to_numeric(df_audit[col], errors='coerce')
    
    # Identify newly introduced NaNs (values that were NOT NaN before, but became NaN)
    newly_coerced = was_not_null & converted_series.isna()
    num_coerced = newly_coerced.sum()
    
    if num_coerced > 0:
        coerced_nan_counts[col] = num_coerced
        # Capture a few examples of what strings were coerced into NaN
        unique_samples = df_audit.loc[newly_coerced, col].unique()[:5].tolist()
        coerced_sample_values[col] = unique_samples

df_coercion_report = pd.DataFrame({
    "Feature": list(coerced_nan_counts.keys()),
    "Values Converted to NaN": list(coerced_nan_counts.values()),
    "Sample Raw Strings Coerced": [coerced_sample_values[k] for k in coerced_nan_counts.keys()]
})

print("==================== STEP 3 COERCION AUDIT ====================")
print(f"Total columns where pd.to_numeric produced new NaNs: {len(coerced_nan_counts)}")
display(df_coercion_report)
print("===============================================================")

==================== STEP 3 COERCION AUDIT ====================
Total columns where pd.to_numeric produced new NaNs: 0


,Feature,Values Converted to NaN,Sample Raw Strings Coerced


In [8]:
# 1. Convert all non-label columns to numeric (which preserves numeric np.inf)
for col in df_audit.columns:
    if col != 'Label':
        df_audit[col] = pd.to_numeric(df_audit[col], errors='coerce')

# 2. Count actual mathematical np.inf values per column
feature_cols = [c for c in df_audit.columns if c != 'Label']
inf_counts = {col: np.isinf(df_audit[col]).sum() for col in feature_cols if np.isinf(df_audit[col]).sum() > 0}

print("==================== MATHEMATICAL INFINITY AUDIT ====================")
print(f"Columns containing mathematical Inf: {len(inf_counts)}")
for col, cnt in inf_counts.items():
    print(f"  -> {col}: {cnt:,} Inf values")

# 3. Separate completely empty padding rows from actual data rows
is_empty_row = df_raw.isna().all(axis=1)

# Mask for rows with NaN or Inf in the actual data section
df_cleanable = df_audit.replace([np.inf, -np.inf], np.nan)
row_has_invalid = df_cleanable.isna().any(axis=1)

# Non-empty rows that were dropped
non_empty_dropped_mask = row_has_invalid & (~is_empty_row)
dropped_real_rows = df_raw.loc[non_empty_dropped_mask]

print("\n==================== DROPPED REAL ROWS AUDIT ====================")
print(f"Total rows dropped overall:                    {row_has_invalid.sum():,}")
print(f"  -> Phantom empty rows dropped:               {is_empty_row.sum():,}")
print(f"  -> Actual data rows dropped:                 {len(dropped_real_rows):,}")
print("\nClass distribution of the actual data rows dropped:")
print(dropped_real_rows['Label'].value_counts(dropna=False))
print("==================================================================")

==================== MATHEMATICAL INFINITY AUDIT ====================
Columns containing mathematical Inf: 2
  -> Flow Bytes/s: 115 Inf values
  -> Flow Packets/s: 135 Inf values

==================== DROPPED REAL ROWS AUDIT ====================
Total rows dropped overall:                    288,737
  -> Phantom empty rows dropped:               288,602
  -> Actual data rows dropped:                 135

Class distribution of the actual data rows dropped:
Label
BENIGN    135
Name: count, dtype: int64


In [10]:
# 1. Build the clean dataset strictly matching your pipeline
df_final = df_cleanable.dropna().copy()

# 2. Extract unique labels present in the non-empty raw rows
raw_real_labels = df_raw.loc[~is_empty_row, 'Label'].value_counts()
clean_labels = df_final['Label'].value_counts()

# 3. Build dynamic label distribution rows directly from actual unique labels
label_rows = []
for label_name in raw_real_labels.index:
    raw_cnt = raw_real_labels.get(label_name, 0)
    clean_cnt = clean_labels.get(label_name, 0)
    label_rows.append({
        "Label": label_name,
        "Raw Valid Rows": raw_cnt,
        "After Cleaning": clean_cnt,
        "Rows Dropped": raw_cnt - clean_cnt,
        "Drop Rate (%)": round(((raw_cnt - clean_cnt) / raw_cnt) * 100, 4)
    })

df_label_audit = pd.DataFrame(label_rows)

print("=================== ACCURATE LABEL AUDIT TABLE ===================")
display(df_label_audit)

# 4. Corrected supervisor audit summary table
audit_summary = [
    ("Raw number of rows", f"{len(df_raw):,}"),
    ("Raw number of columns", f"{df_raw.shape[1]}"),
    ("Phantom empty rows (CSV padding)", f"{is_empty_row.sum():,}"),
    ("Raw valid data rows", f"{(~is_empty_row).sum():,}"),
    ("Number of NaNs before numerical conversion (in valid data)", f"{df_raw.loc[~is_empty_row].isna().sum().sum():,}"),
    ("Number of values converted to NaN by pd.to_numeric", "0"),
    ("Number of Infinity values (Flow Bytes/s & Flow Packets/s)", f"{sum(inf_counts.values()):,}"),
    ("Number of total rows removed", f"{row_has_invalid.sum():,}"),
    ("Number of valid data rows removed", f"{len(dropped_real_rows):,}")
]

# Dynamically append every label so no hardcoded string mismatches occur
for _, r in df_label_audit.iterrows():
    audit_summary.append((f"Label count before cleaning ({r['Label']})", f"{r['Raw Valid Rows']:,}"))
    audit_summary.append((f"Label count after cleaning ({r['Label']})", f"{r['After Cleaning']:,}"))

df_audit_table_corrected = pd.DataFrame(audit_summary, columns=["Metric", "Value"])

print("\n=================== CORRECTED SUPERVISOR AUDIT REPORT ===================")
display(df_audit_table_corrected)

=================== ACCURATE LABEL AUDIT TABLE ===================


,Label,Raw Valid Rows,After Cleaning,Rows Dropped,Drop Rate (%)
0,BENIGN,168186,168051,135,0.0803
1,Web Attack  Brute Force,1507,1507,0,0.0000
2,Web Attack  XSS,652,652,0,0.0000
3,Web Attack  Sql Injection,21,21,0,0.0000



=================== CORRECTED SUPERVISOR AUDIT REPORT ===================


,Metric,Value
0,Raw number of rows,"458,968"
1,Raw number of columns,85
2,Phantom empty rows (CSV padding),"288,602"
3,Raw valid data rows,"170,366"
4,Number of NaNs before numerical conversion (in...,20
5,Number of values converted to NaN by pd.to_num...,0
6,Number of Infinity values (Flow Bytes/s & Flow...,250
7,Number of total rows removed,"288,737"
8,Number of valid data rows removed,135
9,Label count before cleaning (BENIGN),"168,186"
